# 05 — Fish Segmentation for Detection Boxes (Phase 3.5)

Lightweight U-Net on DeepFish segmentation masks (~1.9 M params, CPU-friendly).

In [ ]:
# --- Environment check -------------------------------------------------------
import sys, pathlib
IN_COLAB = "google.colab" in sys.modules
PROJECT = pathlib.Path("/content/reefscan") if IN_COLAB else pathlib.Path.cwd()
print("Colab:", IN_COLAB, "| project:", PROJECT)
if IN_COLAB:
    import os
    from google.colab import drive
    # Mount Drive so datasets/checkpoints survive session resets (optional).
    # drive.mount("/content/drive")


In [ ]:
!python train.py --task seg

In [ ]:
# Visualise predicted boxes on a DeepFish frame
import glob, cv2, matplotlib.pyplot as plt
from src.segmentation.unet import SmallUNet
from src.segmentation.postprocess import masks_to_boxes
import torch

ckpt = torch.load("models/seg/best.pt", map_location="cpu", weights_only=False)
model = SmallUNet(); model.load_state_dict(ckpt["model"]); model.eval()

img_path = sorted(glob.glob("datasets/deepfish/DeepFish/Segmentation/**/*.jpg", recursive=True))[0]
img = cv2.imread(img_path); rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
from src.segmentation.engine_seg import SegmentationTrainer
trainer = SegmentationTrainer(device=torch.device("cpu"))
prob = trainer.predict_mask(rgb)
boxes = masks_to_boxes(prob, threshold=0.5, min_area=400)
for b in boxes:
    cv2.rectangle(img, (b.x1, b.y1), (b.x2, b.y2), (0, 255, 0), 3)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis("off");
